In [3]:
import pandas as pd 
import numpy as np 
import time
from tqdm import tqdm  # ✅ add tqdm
import ollama
import re
import json
import concurrent.futures
import psutil



import threading


In [2]:

# def extract_json(text):
#     # remove code fences if they exist
#     text = re.sub(r"^```[a-zA-Z]*\n?", "", text.strip())  # remove leading ```json or ```
#     text = re.sub(r"```$", "", text.strip())              # remove trailing ```
    
#     # extract first JSON object if multiple
#     match = re.search(r"\{.*\}", text, re.DOTALL)
#     if match:
#         return match.group(0)
#     return text


In [22]:
# dataframe_name = 'DividedDataset/resume_till_100_rows.csv'

In [23]:
# temp_df = pd.read_csv(dataframe_name)

In [1]:
dataframe_name = 'DividedDataset/resume_till_1560_rows.csv'
temp_df = pd.read_csv(dataframe_name)



def extract_json(text):
    # remove code fences if they exist
    text = re.sub(r"^```[a-zA-Z]*\n?", "", text.strip())  # remove leading ```json or ```
    text = re.sub(r"```$", "", text.strip())              # remove trailing ```
    
    # extract first JSON object if multiple
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        return match.group(0)
    return text




atscoretemp = []   # make sure list is initialized
start_time = time.time()

for i in tqdm(range(len(temp_df)), desc="Processing resumes", unit="resume"):
    resume = temp_df.iloc[i,0]
    job    = temp_df.iloc[i,1]

    prompt_content = f"""
    You are an expert resume analyst. Your task is to analyze a candidate's resume and a job description to determine their fit. You MUST ONLY provide a JSON object in your response. Do not include any other text or conversational phrases.

    [JOB DESCRIPTION]
    {job}

    [CANDIDATE RESUME]
    {resume}

    Analyze the candidate's skills and experience against the job description.
    Provide a final match score on a scale from 1 to 100, where 1 is no match and 100 is a perfect match.
    
    Format your output strictly as a JSON object with the following keys:
    {{
      "match_score": int
    }}
    """

    response = ollama.chat(
        model='gemma3',
        messages=[{'role': 'user', 'content': prompt_content}]
    )

    try:
        raw_content = response['message']['content']
        json_str = extract_json(raw_content)   # 🧹 clean response
        ats_score_data = json.loads(json_str)
    
        match_score = ats_score_data.get('match_score', 'N/A')
        atscoretemp.append(match_score)

    except json.JSONDecodeError as e:
        print(f"\nError parsing JSON response at index {i}: {e}")
        print("Raw response:", response['message']['content'])


temp_df['new_ats'] = atscoretemp
temp_df.to_csv(dataframe_name, index=False, encoding='utf-8')

end_time = time.time()
elapsed_time = end_time - start_time
print(f"\n✅ Finished in {elapsed_time:.2f} seconds")


NameError: name 'time' is not defined

In [4]:


# --- Your Existing Code and Imports ---
dataframe_name = "DividedDataset/resume_till_1560_rows.csv"
temp_df = pd.read_csv(dataframe_name)

def extract_json(text):
    # remove code fences if they exist
    text = re.sub(r"^```[a-zA-Z]*\n?", "", text.strip())
    text = re.sub(r"```$", "", text.strip())
    
    # extract first JSON object if multiple
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        return match.group(0)
    return text

# --- The Missing Function ---
def process_single_resume(resume_data):
    i, resume, jd = resume_data
    prompt_content = f"""
    You are an expert resume analyst. Your task is to analyze a candidate's resume and a job description to determine their fit. You MUST ONLY provide a JSON object in your response. Do not include any other text or conversational phrases.

    [JOB DESCRIPTION]
    {jd}

    [CANDIDATE RESUME]
    {resume}

    Analyze the candidate's skills and experience against the job description.
    Provide a final match score on a scale from 1 to 100, where 1 is no match and 100 is a perfect match.
    
    Format your output strictly as a JSON object with the following keys:
    {{
      "match_score": int
    }}
    """
    try:
        response = ollama.chat(
            model='gemma3',
            messages=[{'role': 'user', 'content': prompt_content}]
        )
        raw_content = response['message']['content']
        json_str = extract_json(raw_content)
        ats_score_data = json.loads(json_str)
        match_score = ats_score_data.get('match_score', 'N/A')
        return match_score
    except (json.JSONDecodeError, KeyError) as e:
        # print(f"\nError processing resume at index {i}: {e}") # Optional: Re-add for debugging
        return 'N/A'

# A global event to signal the monitor to stop
stop_event = threading.Event()

def monitor_cpu_daemon(max_cpu_percent=80, check_interval=5):
    """Monitors CPU usage and signals to stop if it exceeds the threshold."""
    print(f"Starting CPU monitor. Will terminate if CPU usage exceeds {max_cpu_percent}%.")
    while not stop_event.is_set():
        cpu_percent = psutil.cpu_percent(interval=1)
        print(f"Current CPU usage: {cpu_percent}%")
        if cpu_percent > max_cpu_percent:
            print(f"CPU usage ({cpu_percent}%) exceeded the limit of {max_cpu_percent}%. Stopping main process.")
            stop_event.set()
        time.sleep(check_interval)
    print("CPU monitor terminated.")

def run_parallel_pipeline_with_monitoring_revised(dataframe, max_workers=8, max_cpu_percent=80):
    start_time = time.time()
    data_to_process = [(i, row['resume'], row['jd']) for i, row in dataframe.iterrows()]
    
    # Start the CPU monitor daemon thread
    monitor_thread = threading.Thread(target=monitor_cpu_daemon, args=(max_cpu_percent,), daemon=True)
    monitor_thread.start()
    
    ats_scores = []
    
    try:
        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
            # Map the processing function to the data
            results = executor.map(process_single_resume, data_to_process)
            
            # Use tqdm to show progress as results are completed
            ats_scores = list(tqdm(results, total=len(data_to_process), desc="Processing Resumes"))
            
    except Exception as e:
        print(f"Main process interrupted due to: {e}")
    finally:
        # Signal the monitor to stop and wait for it
        stop_event.set()
        monitor_thread.join()

    dataframe['new_ats'] = ats_scores
    dataframe.to_csv(dataframe_name, index=False, encoding='utf-8')

    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"\n✅ Finished in {elapsed_time:.2f} seconds")

# --- Call the function with your dataframe ---
run_parallel_pipeline_with_monitoring_revised(temp_df)

Starting CPU monitor. Will terminate if CPU usage exceeds 80%.


Processing Resumes:   0%|                                                                       | 0/60 [00:00<?, ?it/s]

Current CPU usage: 39.8%
Current CPU usage: 51.2%
Current CPU usage: 50.3%
Current CPU usage: 50.8%
Current CPU usage: 50.4%
Current CPU usage: 53.2%


Processing Resumes:   2%|█                                                              | 1/60 [00:36<36:02, 36.65s/it]

Current CPU usage: 52.1%
Current CPU usage: 50.4%
Current CPU usage: 52.2%
Current CPU usage: 50.2%
Current CPU usage: 56.5%
Current CPU usage: 51.8%
Current CPU usage: 51.6%
Current CPU usage: 51.0%
Current CPU usage: 51.2%
Current CPU usage: 51.5%
Current CPU usage: 52.0%
Current CPU usage: 50.6%
Current CPU usage: 56.2%
Current CPU usage: 50.9%
Current CPU usage: 52.1%
Current CPU usage: 51.9%
Current CPU usage: 51.3%
Current CPU usage: 50.3%
Current CPU usage: 50.2%


...

Current CPU usage: 50.3%
Current CPU usage: 53.2%
Current CPU usage: 50.5%
Current CPU usage: 50.9%
Current CPU usage: 50.3%


Processing Resumes:  98%|████████████████████████████████████████████████████████████▉ | 59/60 [36:51<00:28, 28.28s/it]

Current CPU usage: 50.5%
Current CPU usage: 53.1%
Current CPU usage: 50.5%
Current CPU usage: 50.5%
Current CPU usage: 50.5%
Current CPU usage: 50.3%


Processing Resumes: 100%|██████████████████████████████████████████████████████████████| 60/60 [37:28<00:00, 37.48s/it]


CPU monitor terminated.

✅ Finished in 2250.70 seconds


In [5]:
neww_df = pd.read_csv("DividedDataset/resume_till_1560_rows.csv")
neww_df

,resume,jd,ats_score,new_ats
0,Vishal Kumar \nFull Stack Developer \nMy goal ...,Full job description\nPosition: Junior PHP Dev...,51.84,78
1,Vishal Kumar \nFull Stack Developer \nMy goal ...,Full job description\nJob Summary:\n\nWe are s...,23.41,65
2,Vishal Kumar \nFull Stack Developer \nMy goal ...,Job Title: Sr. Website UI/UX Designer and Deve...,49.52,68
3,Vishal Kumar \nFull Stack Developer \nMy goal ...,We are looking for a passionate and enthusiast...,41.10,85
4,Vishal Kumar \nFull Stack Developer \nMy goal ...,Full job description\nJob Title: Software Engi...,32.23,85
5,Vishal Kumar \nFull Stack Developer \nMy goal ...,Software Tester Intern\n\nAbout Us\n\nWe are ...,29.79,65
6,Vishal Kumar \nFull Stack Developer \nMy goal ...,About Warpfy\n\nWe are a fast-growing cross-bo...,39.58,85
7,Vishal Kumar \nFull Stack Developer \nMy goal ...,Full job description\nWork from office only\n\...,38.12,75
8,Vishal Kumar \nFull Stack Developer \nMy goal ...,Company: Weskill Edutech Private Limited\nLoca...,39.01,65
9,Vishal Kumar \nFull Stack Developer \nMy goal ...,Embark on a transformative journey as a Data A...,18.45,65


In [8]:
temp_df.head()

,resume,jd,ats_score
0,Vishal Kumar \nFull Stack Developer \nMy goal ...,Full job description\nPosition: Junior PHP Dev...,51.84
1,Vishal Kumar \nFull Stack Developer \nMy goal ...,Full job description\nJob Summary:\n\nWe are s...,23.41
2,Vishal Kumar \nFull Stack Developer \nMy goal ...,Job Title: Sr. Website UI/UX Designer and Deve...,49.52
3,Vishal Kumar \nFull Stack Developer \nMy goal ...,We are looking for a passionate and enthusiast...,41.10
4,Vishal Kumar \nFull Stack Developer \nMy goal ...,Full job description\nJob Title: Software Engi...,32.23
